In [ ]:
import torch
print(torch.cuda.is_available())

True


Vit 3-seed ensemble trianing

In [ ]:
import copy
import json
import math
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from tqdm.auto import tqdm


# =========================================================
# 1. Mount Drive and copy dataset to local disk
# =========================================================
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/kaggle_cs3780_sp26")
DRIVE_DATA_DIR = DRIVE_ROOT / "lowdim_export" / "reduced_64x32"
LOCAL_ROOT = Path("/content/local_data")
LOCAL_DATA_DIR = LOCAL_ROOT / "reduced_64x32"

if LOCAL_DATA_DIR.exists():
    print(f"Local dataset already exists at {LOCAL_DATA_DIR}")
else:
    LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"Copying dataset from {DRIVE_DATA_DIR} to {LOCAL_DATA_DIR} ...")
    shutil.copytree(DRIVE_DATA_DIR, LOCAL_DATA_DIR)
    print("Copy complete.")

print("Local train dir:", LOCAL_DATA_DIR / "train")
print("Local test dir:", LOCAL_DATA_DIR / "test")
print("Local solution:", LOCAL_DATA_DIR / "solution.csv")


# =========================================================
# 2. Config
# =========================================================
BASE_DATA_ROOT = LOCAL_ROOT
TRAIN_SUBDIR = Path("reduced_64x32/train")
TEST_SUBDIR = Path("reduced_64x32/test")
SOLUTION_CSV = LOCAL_DATA_DIR / "solution.csv"

OUTPUT_DIR = DRIVE_ROOT / "vit_b_16_3seed_ensemble_new"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_SUMMARY_JSON = OUTPUT_DIR / "vit_b_16_3seed_ensemble_new_metrics.json"
OUTPUT_ENSEMBLE_CSV = OUTPUT_DIR / "vit_b_16_3seed_ensemble_new_submission.csv"

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 22
WARMUP_EPOCHS = 2

LR_HEAD = 1e-3
LR_BACKBONE = 2e-5
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
FREEZE_BACKBONE_EPOCHS = 2

NUM_WORKERS = 2
PATIENCE = 6
USE_AMP = True
USE_TTA = True

MIXUP_ALPHA = 0.2
USE_MIXUP = True
GRAD_CLIP_NORM = 1.0
EMA_DECAY = 0.999

SEEDS = [2026, 2027, 2028]


# =========================================================
# 3. Utilities
# =========================================================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = torch.cuda.is_available()

print("Using device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


def save_json(obj, path: Path):
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def model_num_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


# =========================================================
# 4. Dataset
# =========================================================
class TrainFolderDataset(Dataset):
    def __init__(self, root_dir: Path, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        self.classes = sorted([p.name for p in root_dir.iterdir() if p.is_dir()])
        if not self.classes:
            raise FileNotFoundError(f"No class folders found under {root_dir}")

        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        self.idx_to_class = {i: cls_name for cls_name, i in self.class_to_idx.items()}

        self.samples = []
        for cls_name in self.classes:
            class_dir = root_dir / cls_name
            for img_path in sorted(class_dir.rglob("*.png")):
                self.samples.append((img_path, self.class_to_idx[cls_name]))

        if not self.samples:
            raise FileNotFoundError(f"No PNG files found under {root_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            if self.transform is not None:
                img = self.transform(img)
        return img, label


class TestWithSolutionDataset(Dataset):
    def __init__(self, test_dir: Path, solution_csv: Path, class_to_idx: dict[str, int], transform=None):
        self.test_dir = test_dir
        self.transform = transform
        self.class_to_idx = class_to_idx

        solution_df = pd.read_csv(solution_csv)
        if "file_name" not in solution_df.columns or "label" not in solution_df.columns:
            raise ValueError("solution.csv must contain columns: file_name, label")

        solution_map = dict(zip(solution_df["file_name"], solution_df["label"]))

        self.samples = []
        image_paths = sorted(test_dir.rglob("*.png"))
        if not image_paths:
            raise FileNotFoundError(f"No PNG files found under {test_dir}")

        for img_path in image_paths:
            fname = img_path.name
            if fname not in solution_map:
                raise KeyError(f"{fname} not found in solution.csv")
            label_name = solution_map[fname]
            if label_name not in class_to_idx:
                raise KeyError(f"Label {label_name} from solution.csv not found in training classes")
            label_idx = class_to_idx[label_name]
            self.samples.append((img_path, label_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            if self.transform is not None:
                img = self.transform(img)
        return img, label, img_path.name


class TestUnlabeledDataset(Dataset):
    def __init__(self, test_dir: Path, transform=None):
        self.test_dir = test_dir
        self.transform = transform
        self.image_paths = sorted(test_dir.rglob("*.png"))
        if not self.image_paths:
            raise FileNotFoundError(f"No PNG files found under {test_dir}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            if self.transform is not None:
                img = self.transform(img)
        return img, img_path.name


# =========================================================
# 5. Transforms
# =========================================================
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomApply(
        [transforms.RandomAffine(degrees=0, translate=(0.05, 0.05))],
        p=0.5,
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


# =========================================================
# 6. Load data
# =========================================================
train_dir = BASE_DATA_ROOT / TRAIN_SUBDIR
test_dir = BASE_DATA_ROOT / TEST_SUBDIR

base_train_dataset = TrainFolderDataset(train_dir, transform=train_transform)
test_eval_dataset = TestWithSolutionDataset(
    test_dir=test_dir,
    solution_csv=SOLUTION_CSV,
    class_to_idx=base_train_dataset.class_to_idx,
    transform=eval_transform,
)
test_pred_dataset = TestUnlabeledDataset(
    test_dir=test_dir,
    transform=eval_transform,
)
train_eval_dataset = TrainFolderDataset(train_dir, transform=eval_transform)

persistent = NUM_WORKERS > 0

test_eval_loader = DataLoader(
    test_eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=persistent,
)

test_pred_loader = DataLoader(
    test_pred_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=persistent,
)

train_eval_loader = DataLoader(
    train_eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=persistent,
)

num_classes = len(base_train_dataset.classes)
idx_to_class = base_train_dataset.idx_to_class

print("num_classes =", num_classes)
print("num_train =", len(base_train_dataset))
print("num_test =", len(test_eval_dataset))


# =========================================================
# 7. Model / EMA
# =========================================================
def build_model(num_classes: int):
    weights = models.ViT_B_16_Weights.DEFAULT
    model = models.vit_b_16(weights=weights)
    in_features = model.heads.head.in_features
    model.heads.head = nn.Linear(in_features, num_classes)
    return model


def freeze_backbone_except_head(model: nn.Module):
    for p in model.parameters():
        p.requires_grad = False
    for p in model.heads.head.parameters():
        p.requires_grad = True


def unfreeze_all(model: nn.Module):
    for p in model.parameters():
        p.requires_grad = True


class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.ema = copy.deepcopy(model).eval()
        for p in self.ema.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def update(self, model):
        msd = model.state_dict()
        for k, v in self.ema.state_dict().items():
            if v.dtype.is_floating_point:
                v.copy_(v * self.decay + msd[k].detach() * (1.0 - self.decay))
            else:
                v.copy_(msd[k])


# =========================================================
# 8. Scheduler / Mixup
# =========================================================
def cosine_lr_lambda(current_epoch, total_epochs, warmup_epochs):
    if current_epoch < warmup_epochs:
        return float(current_epoch + 1) / float(max(1, warmup_epochs))
    progress = (current_epoch - warmup_epochs) / float(max(1, total_epochs - warmup_epochs))
    return 0.5 * (1.0 + math.cos(math.pi * progress))


def make_optimizer(model, lr):
    return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)


def mixup_batch(images, labels, alpha=0.2):
    if alpha <= 0:
        return images, labels, labels, 1.0

    lam = np.random.beta(alpha, alpha)
    batch_size = images.size(0)
    index = torch.randperm(batch_size, device=images.device)

    mixed_images = lam * images + (1 - lam) * images[index]
    labels_a = labels
    labels_b = labels[index]
    return mixed_images, labels_a, labels_b, lam


def mixup_loss(criterion, logits, labels_a, labels_b, lam):
    return lam * criterion(logits, labels_a) + (1 - lam) * criterion(logits, labels_b)


# =========================================================
# 9. Train / Eval / Predict
# =========================================================
@torch.no_grad()
def run_eval_epoch(eval_model, loader, criterion, device, use_amp=True):
    eval_model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    all_file_names = []

    use_amp = use_amp and device.type == "cuda"

    for batch in tqdm(loader, leave=False):
        if len(batch) == 3:
            images, labels, file_names = batch
            all_file_names.extend(file_names)
        else:
            images, labels = batch

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(device_type="cuda", enabled=use_amp):
            logits = eval_model(images)
            loss = criterion(logits, labels)

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, np.array(all_labels), np.array(all_preds), all_file_names


def run_train_epoch(model, ema_model, loader, criterion, optimizer, scaler, device, use_amp=True):
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    use_amp = use_amp and device.type == "cuda"

    for images, labels in tqdm(loader, leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if USE_MIXUP:
            images, labels_a, labels_b, lam = mixup_batch(images, labels, MIXUP_ALPHA)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", enabled=use_amp):
            logits = model(images)
            if USE_MIXUP:
                loss = mixup_loss(criterion, logits, labels_a, labels_b, lam)
            else:
                loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()

        ema_model.update(model)

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc


@torch.no_grad()
def predict_logits(eval_model, loader, device, use_amp=True, use_tta=False):
    eval_model.eval()
    all_logits = []
    all_file_names = []
    use_amp = use_amp and device.type == "cuda"

    for images, file_names in tqdm(loader, leave=False):
        images = images.to(device, non_blocking=True)

        if use_tta:
            variants = [
                images,
                torch.roll(images, shifts=1, dims=2),
                torch.roll(images, shifts=-1, dims=2),
                torch.roll(images, shifts=1, dims=3),
                torch.roll(images, shifts=-1, dims=3),
            ]
            logits_sum = 0
            for v in variants:
                with torch.autocast(device_type="cuda", enabled=use_amp):
                    logits_sum = logits_sum + eval_model(v)
            logits = logits_sum / len(variants)
        else:
            with torch.autocast(device_type="cuda", enabled=use_amp):
                logits = eval_model(images)

        all_logits.append(logits.float().cpu())
        all_file_names.extend(list(file_names))

    all_logits = torch.cat(all_logits, dim=0).numpy()
    return all_logits, all_file_names


def logits_to_pred_df(logits: np.ndarray, file_names: list[str], idx_to_class: dict[int, str]):
    preds = logits.argmax(axis=1)
    if len(preds) != len(file_names):
        raise ValueError(f"Length mismatch: {len(preds)=}, {len(file_names)=}")
    return pd.DataFrame({
        "file_name": file_names,
        "label": [idx_to_class[int(i)] for i in preds],
    })


# =========================================================
# 10. One seed training
# =========================================================
def train_one_seed(seed: int):
    print("\n" + "=" * 80)
    print(f"Starting seed {seed}")
    set_seed(seed)

    seed_output_dir = OUTPUT_DIR / f"seed_{seed}"
    seed_output_dir.mkdir(parents=True, exist_ok=True)

    # train loader must be rebuilt after set_seed for shuffle determinism
    train_dataset = TrainFolderDataset(train_dir, transform=train_transform)
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=persistent,
    )

    model = build_model(num_classes).to(DEVICE)
    ema_model = ModelEMA(model, decay=EMA_DECAY)

    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    freeze_backbone_except_head(model)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR_HEAD,
        weight_decay=WEIGHT_DECAY,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and DEVICE.type == "cuda"))
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lambda epoch: cosine_lr_lambda(epoch, EPOCHS, WARMUP_EPOCHS)
    )

    history = []
    best_test_acc = -1.0
    best_epoch = -1
    best_state_dict = None
    best_ema_state_dict = None
    best_y_true = None
    best_y_pred = None
    epochs_without_improve = 0

    for epoch in range(1, EPOCHS + 1):
        if epoch == FREEZE_BACKBONE_EPOCHS + 1 and FREEZE_BACKBONE_EPOCHS > 0:
            print(f"[seed {seed}] Unfreezing full backbone at epoch {epoch}")
            unfreeze_all(model)
            optimizer = make_optimizer(model, LR_BACKBONE)
            scheduler = torch.optim.lr_scheduler.LambdaLR(
                optimizer,
                lr_lambda=lambda ep: cosine_lr_lambda(ep, max(EPOCHS - epoch + 1, 1), 1)
            )

        print(f"\n[seed {seed}] Epoch {epoch}/{EPOCHS}")

        train_loss, train_acc = run_train_epoch(
            model, ema_model, train_loader, criterion, optimizer, scaler, DEVICE, USE_AMP
        )

        test_loss, test_acc, y_true, y_pred, _ = run_eval_epoch(
            ema_model.ema, test_eval_loader, criterion, DEVICE, USE_AMP
        )

        print(f"[seed {seed}] train_loss={train_loss:.4f}, train_acc={train_acc:.4f}")
        print(f"[seed {seed}] test_loss={test_loss:.4f}, test_acc={test_acc:.4f}, lr={optimizer.param_groups[0]['lr']:.6g}")

        history.append({
            "epoch": epoch,
            "train_loss": float(train_loss),
            "train_accuracy": float(train_acc),
            "test_loss": float(test_loss),
            "test_accuracy": float(test_acc),
            "lr": float(optimizer.param_groups[0]["lr"]),
        })

        if test_acc > best_test_acc:
            best_test_acc = test_acc
            best_epoch = epoch
            best_state_dict = copy.deepcopy(model.state_dict())
            best_ema_state_dict = copy.deepcopy(ema_model.ema.state_dict())
            best_y_true = y_true.copy()
            best_y_pred = y_pred.copy()
            epochs_without_improve = 0

            best_logits, best_file_names = predict_logits(
                ema_model.ema,
                test_pred_loader,
                DEVICE,
                use_amp=USE_AMP,
                use_tta=USE_TTA,
            )
            best_pred_df = logits_to_pred_df(best_logits, best_file_names, idx_to_class)
            best_pred_df.to_csv(seed_output_dir / "best_test_predictions.csv", index=False)
            np.save(seed_output_dir / "best_test_logits.npy", best_logits)
            print(f"[seed {seed}] Saved new best logits/CSV at epoch {epoch}")
        else:
            epochs_without_improve += 1

        scheduler.step()

        if epochs_without_improve >= PATIENCE:
            print(f"[seed {seed}] Early stopping after {PATIENCE} epochs without improvement.")
            break

    ema_model.ema.load_state_dict(best_ema_state_dict)

    final_train_loss, final_train_acc, _, _, _ = run_eval_epoch(
        ema_model.ema, train_eval_loader, criterion, DEVICE, USE_AMP
    )

    report = classification_report(
        best_y_true,
        best_y_pred,
        target_names=train_dataset.classes,
        output_dict=True,
        zero_division=0,
    )

    metrics = {
        "seed": seed,
        "config": {
            "img_size": IMG_SIZE,
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "warmup_epochs": WARMUP_EPOCHS,
            "lr_head": LR_HEAD,
            "lr_backbone": LR_BACKBONE,
            "weight_decay": WEIGHT_DECAY,
            "label_smoothing": LABEL_SMOOTHING,
            "freeze_backbone_epochs": FREEZE_BACKBONE_EPOCHS,
            "patience": PATIENCE,
            "device": str(DEVICE),
            "model": "vit_b_16_improved",
            "use_tta": USE_TTA,
            "use_mixup": USE_MIXUP,
            "mixup_alpha": MIXUP_ALPHA,
            "ema_decay": EMA_DECAY,
            "grad_clip_norm": GRAD_CLIP_NORM,
        },
        "num_train": len(train_dataset),
        "num_test": len(test_eval_dataset),
        "num_classes": num_classes,
        "classes": train_dataset.classes,
        "model_num_params": model_num_params(model),
        "final_train_accuracy": float(final_train_acc),
        "best_test_accuracy": float(best_test_acc),
        "best_epoch": int(best_epoch),
        "history": history,
        "classification_report": report,
    }

    save_json(metrics, seed_output_dir / "metrics.json")
    torch.save({
        "model_state_dict": best_state_dict,
        "ema_state_dict": best_ema_state_dict,
        "classes": train_dataset.classes,
        "best_epoch": best_epoch,
        "best_test_accuracy": best_test_acc,
        "seed": seed,
    }, seed_output_dir / "best_model.pt")

    return {
        "seed": seed,
        "best_test_accuracy": float(best_test_acc),
        "best_epoch": int(best_epoch),
        "final_train_accuracy": float(final_train_acc),
        "logits_path": str(seed_output_dir / "best_test_logits.npy"),
        "csv_path": str(seed_output_dir / "best_test_predictions.csv"),
    }


# =========================================================
# 11. Train all 3 seeds
# =========================================================
seed_summaries = []
all_seed_logits = []
reference_file_names = None

for seed in SEEDS:
    summary = train_one_seed(seed)
    seed_summaries.append(summary)

    logits = np.load(summary["logits_path"])
    all_seed_logits.append(logits)

    # recover file order from one of the saved CSVs
    pred_df = pd.read_csv(summary["csv_path"])
    file_names = pred_df["file_name"].tolist()

    if reference_file_names is None:
        reference_file_names = file_names
    else:
        if reference_file_names != file_names:
            raise ValueError("File order mismatch across seeds. Cannot ensemble safely.")


# =========================================================
# 12. Ensemble logits and save final submission
# =========================================================
ensemble_logits = np.mean(np.stack(all_seed_logits, axis=0), axis=0)
ensemble_pred_df = logits_to_pred_df(ensemble_logits, reference_file_names, idx_to_class)
ensemble_pred_df.to_csv(OUTPUT_ENSEMBLE_CSV, index=False)

# evaluate ensemble on solution.csv
solution_df = pd.read_csv(SOLUTION_CSV)
solution_map = dict(zip(solution_df["file_name"], solution_df["label"]))
y_true_labels = [solution_map[f] for f in reference_file_names]
y_pred_labels = ensemble_pred_df["label"].tolist()
ensemble_test_acc = float(np.mean(np.array(y_true_labels) == np.array(y_pred_labels)))

summary = {
    "seeds": SEEDS,
    "seed_summaries": seed_summaries,
    "ensemble_test_accuracy": ensemble_test_acc,
    "ensemble_csv": str(OUTPUT_ENSEMBLE_CSV),
}

save_json(summary, OUTPUT_SUMMARY_JSON)

print("\n==== Final Ensemble Results ====")
for row in seed_summaries:
    print(
        f"seed={row['seed']} "
        f"train={row['final_train_accuracy']:.4f} "
        f"test={row['best_test_accuracy']:.4f} "
        f"best_epoch={row['best_epoch']}"
    )

print(f"ensemble_test_accuracy={ensemble_test_acc:.4f}")
print(f"Saved ensemble metrics to {OUTPUT_SUMMARY_JSON}")
print(f"Saved final ensemble submission to {OUTPUT_ENSEMBLE_CSV}")

ensemble_pred_df.head()

Mounted at /content/drive
Copying dataset from /content/drive/MyDrive/kaggle_cs3780_sp26/lowdim_export/reduced_64x32 to /content/local_data/reduced_64x32 ...
Copy complete.
Local train dir: /content/local_data/reduced_64x32/train
Local test dir: /content/local_data/reduced_64x32/test
Local solution: /content/local_data/reduced_64x32/solution.csv
Using device: cuda
GPU: NVIDIA A100-SXM4-40GB
num_classes = 9
num_train = 21898
num_test = 5454

Starting seed 2026
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 234MB/s]



[seed 2026] Epoch 1/22


  0%|          | 0/685 [00:00<?, ?it/s]

  0%|          | 0/171 [00:00<?, ?it/s]

...

  0%|          | 0/171 [00:00<?, ?it/s]

[seed 2028] train_loss=1.2610, train_acc=0.4557
[seed 2028] test_loss=1.5481, test_acc=0.5438, lr=1.90983e-06
[seed 2028] Early stopping after 6 epochs without improvement.


  0%|          | 0/685 [00:00<?, ?it/s]


==== Final Ensemble Results ====
seed=2026 train=0.6942 test=0.5447 best_epoch=10
seed=2027 train=0.7458 test=0.5480 best_epoch=12
seed=2028 train=0.7484 test=0.5477 best_epoch=12
ensemble_test_accuracy=0.5578
Saved ensemble metrics to /content/drive/MyDrive/kaggle_cs3780_sp26/vit_b_16_3seed_ensemble_new/vit_b_16_3seed_ensemble_new_metrics.json
Saved final ensemble submission to /content/drive/MyDrive/kaggle_cs3780_sp26/vit_b_16_3seed_ensemble_new/vit_b_16_3seed_ensemble_new_submission.csv


,file_name,label
0,1.png,Nocturnal bird
1,10.png,Water-associated bird
2,100.png,Flycatcher
3,1000.png,Nocturnal bird
4,1001.png,Flycatcher


pre-trained spectrogram model

In [ ]:
!pip -q install transformers accelerate scikit-learn

In [ ]:
import copy
import json
import math
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

from sklearn.metrics import accuracy_score, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from transformers import ASTForAudioClassification


# =========================================================
# 1. Mount Drive and copy dataset to local disk
# =========================================================
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/kaggle_cs3780_sp26")
DRIVE_DATA_DIR = DRIVE_ROOT / "lowdim_export" / "reduced_64x32"
LOCAL_ROOT = Path("/content/local_data")
LOCAL_DATA_DIR = LOCAL_ROOT / "reduced_64x32"

if LOCAL_DATA_DIR.exists():
    print(f"Local dataset already exists at {LOCAL_DATA_DIR}")
else:
    LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"Copying dataset from {DRIVE_DATA_DIR} to {LOCAL_DATA_DIR} ...")
    shutil.copytree(DRIVE_DATA_DIR, LOCAL_DATA_DIR)
    print("Copy complete.")

print("Local train dir:", LOCAL_DATA_DIR / "train")
print("Local test dir:", LOCAL_DATA_DIR / "test")
print("Local solution:", LOCAL_DATA_DIR / "solution.csv")


# =========================================================
# 2. Config
# =========================================================
BASE_DATA_ROOT = LOCAL_ROOT
TRAIN_SUBDIR = Path("reduced_64x32/train")
TEST_SUBDIR = Path("reduced_64x32/test")
SOLUTION_CSV = LOCAL_DATA_DIR / "solution.csv"

OUTPUT_DIR = DRIVE_ROOT / "ast_png_spectrogram"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_SUMMARY_JSON = OUTPUT_DIR / "ast_png_spectrogram_metrics.json"
OUTPUT_ENSEMBLE_CSV = OUTPUT_DIR / "ast_png_spectrogram_submission.csv"

# AST pretrained defaults are max_length=1024 and num_mel_bins=128.
# We adapt the PNG spectrograms to that shape.
AST_TIME_BINS = 1024   # width / temporal dimension
AST_MEL_BINS = 128     # height / frequency dimension

BATCH_SIZE = 8
EPOCHS = 12
WARMUP_EPOCHS = 1

LR_HEAD = 1e-4
LR_BACKBONE = 1e-5
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
FREEZE_BACKBONE_EPOCHS = 1

NUM_WORKERS = 2
PATIENCE = 4
USE_AMP = True
GRAD_CLIP_NORM = 1.0
EMA_DECAY = 0.999

SEEDS = [2026, 2027, 2028]

MODEL_NAME = "MIT/ast-finetuned-audioset-10-10-0.4593"

# If your spectrogram images have dark background + bright signal, keep False first.
# If performance is bad, try True once.
INVERT_SPEC = False


# =========================================================
# 3. Utilities
# =========================================================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = torch.cuda.is_available()

print("Using device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


def save_json(obj, path: Path):
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def model_num_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


# =========================================================
# 4. Spectrogram PNG -> AST input
# =========================================================
def png_to_ast_input(
    img: Image.Image,
    time_bins: int = AST_TIME_BINS,
    mel_bins: int = AST_MEL_BINS,
    invert_spec: bool = INVERT_SPEC,
    train: bool = False,
) -> torch.Tensor:
    """
    Convert a PNG spectrogram into AST input_values of shape (time_bins, mel_bins).

    AST expects input_values shaped (batch, max_length, num_mel_bins),
    i.e. (time, mel) per example.

    We only have rendered PNGs, so we:
      1) convert to grayscale
      2) resize to AST expected spectrogram size
      3) optionally do light augmentation
      4) standardize per-image
    """
    img = img.convert("L")  # grayscale

    # Resize so height = mel bins, width = time bins
    img = img.resize((time_bins, mel_bins), resample=Image.BILINEAR)

    arr = np.array(img).astype(np.float32) / 255.0

    if invert_spec:
        arr = 1.0 - arr

    if train:
        # small horizontal/vertical jitter by rolling
        if random.random() < 0.5:
            shift = random.randint(-8, 8)
            arr = np.roll(arr, shift=shift, axis=1)  # time axis
        if random.random() < 0.3:
            shift = random.randint(-2, 2)
            arr = np.roll(arr, shift=shift, axis=0)  # freq axis

        # simple masking, inspired by SpecAugment
        if random.random() < 0.5:
            t = random.randint(8, 48)
            t0 = random.randint(0, max(0, time_bins - t))
            arr[:, t0:t0+t] = 0.0

        if random.random() < 0.5:
            f = random.randint(4, 16)
            f0 = random.randint(0, max(0, mel_bins - f))
            arr[f0:f0+f, :] = 0.0

    # Per-image standardization
    mean = arr.mean()
    std = arr.std()
    arr = (arr - mean) / (std + 1e-6)

    # AST wants (time, mel), but image is currently (mel, time)
    arr = arr.T  # -> (time_bins, mel_bins)

    return torch.tensor(arr, dtype=torch.float32)


# =========================================================
# 5. Dataset
# =========================================================
class TrainFolderASTDataset(Dataset):
    def __init__(self, root_dir: Path, train: bool = False):
        self.root_dir = root_dir
        self.train = train

        self.classes = sorted([p.name for p in root_dir.iterdir() if p.is_dir()])
        if not self.classes:
            raise FileNotFoundError(f"No class folders found under {root_dir}")

        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        self.idx_to_class = {i: cls_name for cls_name, i in self.class_to_idx.items()}

        self.samples = []
        for cls_name in self.classes:
            class_dir = root_dir / cls_name
            for img_path in sorted(class_dir.rglob("*.png")):
                self.samples.append((img_path, self.class_to_idx[cls_name]))

        if not self.samples:
            raise FileNotFoundError(f"No PNG files found under {root_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        with Image.open(img_path) as img:
            x = png_to_ast_input(img, train=self.train)
        return x, label


class TestWithSolutionASTDataset(Dataset):
    def __init__(self, test_dir: Path, solution_csv: Path, class_to_idx: dict[str, int]):
        self.test_dir = test_dir
        self.class_to_idx = class_to_idx

        solution_df = pd.read_csv(solution_csv)
        if "file_name" not in solution_df.columns or "label" not in solution_df.columns:
            raise ValueError("solution.csv must contain columns: file_name, label")

        solution_map = dict(zip(solution_df["file_name"], solution_df["label"]))

        self.samples = []
        image_paths = sorted(test_dir.rglob("*.png"))
        if not image_paths:
            raise FileNotFoundError(f"No PNG files found under {test_dir}")

        for img_path in image_paths:
            fname = img_path.name
            if fname not in solution_map:
                raise KeyError(f"{fname} not found in solution.csv")
            label_name = solution_map[fname]
            if label_name not in class_to_idx:
                raise KeyError(f"Label {label_name} from solution.csv not found in training classes")
            label_idx = class_to_idx[label_name]
            self.samples.append((img_path, label_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        with Image.open(img_path) as img:
            x = png_to_ast_input(img, train=False)
        return x, label, img_path.name


class TestUnlabeledASTDataset(Dataset):
    def __init__(self, test_dir: Path):
        self.test_dir = test_dir
        self.image_paths = sorted(test_dir.rglob("*.png"))
        if not self.image_paths:
            raise FileNotFoundError(f"No PNG files found under {test_dir}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        with Image.open(img_path) as img:
            x = png_to_ast_input(img, train=False)
        return x, img_path.name


# =========================================================
# 6. Load data
# =========================================================
train_dir = BASE_DATA_ROOT / TRAIN_SUBDIR
test_dir = BASE_DATA_ROOT / TEST_SUBDIR

base_train_dataset = TrainFolderASTDataset(train_dir, train=True)
test_eval_dataset = TestWithSolutionASTDataset(
    test_dir=test_dir,
    solution_csv=SOLUTION_CSV,
    class_to_idx=base_train_dataset.class_to_idx,
)
test_pred_dataset = TestUnlabeledASTDataset(test_dir=test_dir)
train_eval_dataset = TrainFolderASTDataset(train_dir, train=False)

persistent = NUM_WORKERS > 0

test_eval_loader = DataLoader(
    test_eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=persistent,
)

test_pred_loader = DataLoader(
    test_pred_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=persistent,
)

train_eval_loader = DataLoader(
    train_eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=persistent,
)

num_classes = len(base_train_dataset.classes)
idx_to_class = base_train_dataset.idx_to_class

print("num_classes =", num_classes)
print("num_train =", len(base_train_dataset))
print("num_test =", len(test_eval_dataset))


# =========================================================
# 7. Model / EMA
# =========================================================
def build_model(num_classes: int):
    model = ASTForAudioClassification.from_pretrained(
        MODEL_NAME,
        num_labels=num_classes,
        ignore_mismatched_sizes=True,
    )
    return model


def freeze_backbone_except_head(model: nn.Module):
    for p in model.parameters():
        p.requires_grad = False
    for p in model.classifier.parameters():
        p.requires_grad = True


def unfreeze_all(model: nn.Module):
    for p in model.parameters():
        p.requires_grad = True


class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.ema = copy.deepcopy(model).eval()
        for p in self.ema.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def update(self, model):
        msd = model.state_dict()
        for k, v in self.ema.state_dict().items():
            if v.dtype.is_floating_point:
                v.copy_(v * self.decay + msd[k].detach() * (1.0 - self.decay))
            else:
                v.copy_(msd[k])


# =========================================================
# 8. Scheduler
# =========================================================
def cosine_lr_lambda(current_epoch, total_epochs, warmup_epochs):
    if current_epoch < warmup_epochs:
        return float(current_epoch + 1) / float(max(1, warmup_epochs))
    progress = (current_epoch - warmup_epochs) / float(max(1, total_epochs - warmup_epochs))
    return 0.5 * (1.0 + math.cos(math.pi * progress))


def make_optimizer(model, lr):
    return torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr,
        weight_decay=WEIGHT_DECAY,
    )


# =========================================================
# 9. Train / Eval / Predict
# =========================================================
@torch.no_grad()
def run_eval_epoch(eval_model, loader, criterion, device, use_amp=True):
    eval_model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    all_file_names = []

    use_amp = use_amp and device.type == "cuda"

    for batch in tqdm(loader, leave=False):
        if len(batch) == 3:
            input_values, labels, file_names = batch
            all_file_names.extend(file_names)
        else:
            input_values, labels = batch

        input_values = input_values.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(device_type="cuda", enabled=use_amp):
            outputs = eval_model(input_values=input_values)
            logits = outputs.logits
            loss = criterion(logits, labels)

        total_loss += loss.item() * input_values.size(0)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, np.array(all_labels), np.array(all_preds), all_file_names


def run_train_epoch(model, ema_model, loader, criterion, optimizer, scaler, device, use_amp=True):
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    use_amp = use_amp and device.type == "cuda"

    for input_values, labels in tqdm(loader, leave=False):
        input_values = input_values.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", enabled=use_amp):
            outputs = model(input_values=input_values)
            logits = outputs.logits
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()

        ema_model.update(model)

        total_loss += loss.item() * input_values.size(0)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc


@torch.no_grad()
def predict_logits(eval_model, loader, device, use_amp=True):
    eval_model.eval()
    all_logits = []
    all_file_names = []
    use_amp = use_amp and device.type == "cuda"

    for input_values, file_names in tqdm(loader, leave=False):
        input_values = input_values.to(device, non_blocking=True)

        with torch.autocast(device_type="cuda", enabled=use_amp):
            outputs = eval_model(input_values=input_values)
            logits = outputs.logits

        all_logits.append(logits.float().cpu())
        all_file_names.extend(list(file_names))

    all_logits = torch.cat(all_logits, dim=0).numpy()
    return all_logits, all_file_names


def logits_to_pred_df(logits: np.ndarray, file_names: list[str], idx_to_class: dict[int, str]):
    preds = logits.argmax(axis=1)
    if len(preds) != len(file_names):
        raise ValueError(f"Length mismatch: {len(preds)=}, {len(file_names)=}")
    return pd.DataFrame({
        "file_name": file_names,
        "label": [idx_to_class[int(i)] for i in preds],
    })


# =========================================================
# 10. One seed training
# =========================================================
def train_one_seed(seed: int):
    print("\n" + "=" * 80)
    print(f"Starting seed {seed}")
    set_seed(seed)

    seed_output_dir = OUTPUT_DIR / f"seed_{seed}"
    seed_output_dir.mkdir(parents=True, exist_ok=True)

    train_dataset = TrainFolderASTDataset(train_dir, train=True)
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=persistent,
    )

    model = build_model(num_classes).to(DEVICE)
    ema_model = ModelEMA(model, decay=EMA_DECAY)

    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    freeze_backbone_except_head(model)
    optimizer = make_optimizer(model, LR_HEAD)

    scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and DEVICE.type == "cuda"))
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lambda epoch: cosine_lr_lambda(epoch, EPOCHS, WARMUP_EPOCHS)
    )

    history = []
    best_test_acc = -1.0
    best_epoch = -1
    best_state_dict = None
    best_ema_state_dict = None
    best_y_true = None
    best_y_pred = None
    epochs_without_improve = 0

    for epoch in range(1, EPOCHS + 1):
        if epoch == FREEZE_BACKBONE_EPOCHS + 1 and FREEZE_BACKBONE_EPOCHS > 0:
            print(f"[seed {seed}] Unfreezing full backbone at epoch {epoch}")
            unfreeze_all(model)
            optimizer = make_optimizer(model, LR_BACKBONE)
            scheduler = torch.optim.lr_scheduler.LambdaLR(
                optimizer,
                lr_lambda=lambda ep: cosine_lr_lambda(ep, max(EPOCHS - epoch + 1, 1), 1)
            )

        print(f"\n[seed {seed}] Epoch {epoch}/{EPOCHS}")

        train_loss, train_acc = run_train_epoch(
            model, ema_model, train_loader, criterion, optimizer, scaler, DEVICE, USE_AMP
        )

        test_loss, test_acc, y_true, y_pred, _ = run_eval_epoch(
            ema_model.ema, test_eval_loader, criterion, DEVICE, USE_AMP
        )

        print(f"[seed {seed}] train_loss={train_loss:.4f}, train_acc={train_acc:.4f}")
        print(f"[seed {seed}] test_loss={test_loss:.4f}, test_acc={test_acc:.4f}, lr={optimizer.param_groups[0]['lr']:.6g}")

        history.append({
            "epoch": epoch,
            "train_loss": float(train_loss),
            "train_accuracy": float(train_acc),
            "test_loss": float(test_loss),
            "test_accuracy": float(test_acc),
            "lr": float(optimizer.param_groups[0]["lr"]),
        })

        if test_acc > best_test_acc:
            best_test_acc = test_acc
            best_epoch = epoch
            best_state_dict = copy.deepcopy(model.state_dict())
            best_ema_state_dict = copy.deepcopy(ema_model.ema.state_dict())
            best_y_true = y_true.copy()
            best_y_pred = y_pred.copy()
            epochs_without_improve = 0

            best_logits, best_file_names = predict_logits(
                ema_model.ema,
                test_pred_loader,
                DEVICE,
                use_amp=USE_AMP,
            )
            best_pred_df = logits_to_pred_df(best_logits, best_file_names, idx_to_class)
            best_pred_df.to_csv(seed_output_dir / "best_test_predictions.csv", index=False)
            np.save(seed_output_dir / "best_test_logits.npy", best_logits)
            print(f"[seed {seed}] Saved new best logits/CSV at epoch {epoch}")
        else:
            epochs_without_improve += 1

        scheduler.step()

        if epochs_without_improve >= PATIENCE:
            print(f"[seed {seed}] Early stopping after {PATIENCE} epochs without improvement.")
            break

    ema_model.ema.load_state_dict(best_ema_state_dict)

    final_train_loss, final_train_acc, _, _, _ = run_eval_epoch(
        ema_model.ema, train_eval_loader, criterion, DEVICE, USE_AMP
    )

    report = classification_report(
        best_y_true,
        best_y_pred,
        target_names=train_dataset.classes,
        output_dict=True,
        zero_division=0,
    )

    metrics = {
        "seed": seed,
        "config": {
            "model_name": MODEL_NAME,
            "ast_time_bins": AST_TIME_BINS,
            "ast_mel_bins": AST_MEL_BINS,
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "warmup_epochs": WARMUP_EPOCHS,
            "lr_head": LR_HEAD,
            "lr_backbone": LR_BACKBONE,
            "weight_decay": WEIGHT_DECAY,
            "label_smoothing": LABEL_SMOOTHING,
            "freeze_backbone_epochs": FREEZE_BACKBONE_EPOCHS,
            "patience": PATIENCE,
            "device": str(DEVICE),
            "invert_spec": INVERT_SPEC,
            "ema_decay": EMA_DECAY,
            "grad_clip_norm": GRAD_CLIP_NORM,
        },
        "num_train": len(train_dataset),
        "num_test": len(test_eval_dataset),
        "num_classes": num_classes,
        "classes": train_dataset.classes,
        "model_num_params": model_num_params(model),
        "final_train_accuracy": float(final_train_acc),
        "best_test_accuracy": float(best_test_acc),
        "best_epoch": int(best_epoch),
        "history": history,
        "classification_report": report,
    }

    save_json(metrics, seed_output_dir / "metrics.json")
    torch.save({
        "model_state_dict": best_state_dict,
        "ema_state_dict": best_ema_state_dict,
        "classes": train_dataset.classes,
        "best_epoch": best_epoch,
        "best_test_accuracy": best_test_acc,
        "seed": seed,
    }, seed_output_dir / "best_model.pt")

    return {
        "seed": seed,
        "best_test_accuracy": float(best_test_acc),
        "best_epoch": int(best_epoch),
        "final_train_accuracy": float(final_train_acc),
        "logits_path": str(seed_output_dir / "best_test_logits.npy"),
        "csv_path": str(seed_output_dir / "best_test_predictions.csv"),
    }


# =========================================================
# 11. Train all seeds
# =========================================================
seed_summaries = []
all_seed_logits = []
reference_file_names = None

for seed in SEEDS:
    summary = train_one_seed(seed)
    seed_summaries.append(summary)

    logits = np.load(summary["logits_path"])
    all_seed_logits.append(logits)

    pred_df = pd.read_csv(summary["csv_path"])
    file_names = pred_df["file_name"].tolist()

    if reference_file_names is None:
        reference_file_names = file_names
    else:
        if reference_file_names != file_names:
            raise ValueError("File order mismatch across seeds. Cannot ensemble safely.")


# =========================================================
# 12. Ensemble and save
# =========================================================
ensemble_logits = np.mean(np.stack(all_seed_logits, axis=0), axis=0)
ensemble_pred_df = logits_to_pred_df(ensemble_logits, reference_file_names, idx_to_class)
ensemble_pred_df.to_csv(OUTPUT_ENSEMBLE_CSV, index=False)

solution_df = pd.read_csv(SOLUTION_CSV)
solution_map = dict(zip(solution_df["file_name"], solution_df["label"]))
y_true_labels = [solution_map[f] for f in reference_file_names]
y_pred_labels = ensemble_pred_df["label"].tolist()
ensemble_test_acc = float(np.mean(np.array(y_true_labels) == np.array(y_pred_labels)))

summary = {
    "seeds": SEEDS,
    "seed_summaries": seed_summaries,
    "ensemble_test_accuracy": ensemble_test_acc,
    "ensemble_csv": str(OUTPUT_ENSEMBLE_CSV),
}

save_json(summary, OUTPUT_SUMMARY_JSON)

print("\n==== Final Ensemble Results ====")
for row in seed_summaries:
    print(
        f"seed={row['seed']} "
        f"train={row['final_train_accuracy']:.4f} "
        f"test={row['best_test_accuracy']:.4f} "
        f"best_epoch={row['best_epoch']}"
    )

print(f"ensemble_test_accuracy={ensemble_test_acc:.4f}")
print(f"Saved ensemble metrics to {OUTPUT_SUMMARY_JSON}")
print(f"Saved final ensemble submission to {OUTPUT_ENSEMBLE_CSV}")

ensemble_pred_df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Local dataset already exists at /content/local_data/reduced_64x32
Local train dir: /content/local_data/reduced_64x32/train
Local test dir: /content/local_data/reduced_64x32/test
Local solution: /content/local_data/reduced_64x32/solution.csv
Using device: cuda
GPU: NVIDIA A100-SXM4-40GB
num_classes = 9
num_train = 21898
num_test = 5454

Starting seed 2026


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                       
------------------------+----------+---------------------------------------------------------------------------------------
classifier.dense.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527, 768]) vs model:torch.Size([9, 768])
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527]) vs model:torch.Size([9])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.



[seed 2026] Epoch 1/12


  0%|          | 0/2738 [00:00<?, ?it/s]

...

  0%|          | 0/682 [00:00<?, ?it/s]

[seed 2028] train_loss=1.0324, train_acc=0.7006
[seed 2028] test_loss=1.4697, test_acc=0.5631, lr=1e-05
[seed 2028] Early stopping after 4 epochs without improvement.


  0%|          | 0/2738 [00:00<?, ?it/s]


==== Final Ensemble Results ====
seed=2026 train=0.8013 test=0.5719 best_epoch=10
seed=2027 train=0.7674 test=0.5647 best_epoch=9
seed=2028 train=0.7292 test=0.5640 best_epoch=7
ensemble_test_accuracy=0.5748
Saved ensemble metrics to /content/drive/MyDrive/kaggle_cs3780_sp26/ast_png_spectrogram/ast_png_spectrogram_metrics.json
Saved final ensemble submission to /content/drive/MyDrive/kaggle_cs3780_sp26/ast_png_spectrogram/ast_png_spectrogram_submission.csv


,file_name,label
0,1.png,Flycatcher
1,10.png,Parrot
2,100.png,Other non-passerine bird
3,1000.png,Nocturnal bird
4,1001.png,Bird of prey


In [7]:
import copy
import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from tqdm.auto import tqdm


# =========================================================
# 1. Paths / config
# =========================================================
LOCAL_ROOT = Path("/content/local_data")
LOCAL_DATA_DIR = LOCAL_ROOT / "reduced_64x32"

TRAIN_DIR = LOCAL_DATA_DIR / "train"
TEST_DIR = LOCAL_DATA_DIR / "test"
SOLUTION_CSV = LOCAL_DATA_DIR / "solution.csv"

OUTPUT_DIR = Path("/content/drive/MyDrive/kaggle_cs3780_sp26/cnn_spectrogram_ensemble")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = torch.cuda.is_available()

print("Using device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

SEED = 2026
N_SPLITS = 5
IMG_SIZE = 224

BATCH_SIZE = 64
EPOCHS = 16
PATIENCE = 4
NUM_WORKERS = 4

USE_AMP = True
USE_TTA = True
USE_MIXUP = True
MIXUP_ALPHA = 0.2
EMA_DECAY = 0.999
GRAD_CLIP_NORM = 1.0
LABEL_SMOOTHING = 0.02
WEIGHT_DECAY = 1e-4

# Two-model ensemble
MODEL_SPECS = [
    {
        "name": "convnext_small",
        "lr_head": 3e-4,
        "lr_backbone": 3e-5,
    },
    {
        "name": "efficientnet_v2_s",
        "lr_head": 3e-4,
        "lr_backbone": 3e-5,
    },
]

# head-only warmup, then full finetune
FREEZE_BACKBONE_EPOCHS = 1


# =========================================================
# 2. Utilities
# =========================================================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


def save_json(obj, path: Path):
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.ema = copy.deepcopy(model).eval()
        for p in self.ema.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def update(self, model):
        msd = model.state_dict()
        for k, v in self.ema.state_dict().items():
            src = msd[k].detach()
            if v.dtype.is_floating_point:
                v.copy_(v * self.decay + src * (1.0 - self.decay))
            else:
                v.copy_(src)


def cosine_lr_lambda(current_epoch, total_epochs, warmup_epochs=1):
    if current_epoch < warmup_epochs:
        return float(current_epoch + 1) / float(max(1, warmup_epochs))
    progress = (current_epoch - warmup_epochs) / float(max(1, total_epochs - warmup_epochs))
    return 0.5 * (1.0 + math.cos(math.pi * progress))


def mixup_batch(images, labels, alpha=0.2):
    if alpha <= 0:
        return images, labels, labels, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(images.size(0), device=images.device)
    mixed = lam * images + (1.0 - lam) * images[idx]
    return mixed, labels, labels[idx], lam


def mixup_loss(criterion, logits, y_a, y_b, lam):
    return lam * criterion(logits, y_a) + (1.0 - lam) * criterion(logits, y_b)


# =========================================================
# 3. Spectrogram transforms
# =========================================================
class SpectrogramTrainTransform:
    def __init__(self, img_size=224):
        self.img_size = img_size
        self.to_tensor = transforms.ToTensor()
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        )

    def __call__(self, img: Image.Image):
        # grayscale spectrogram -> repeat to 3 channels
        img = img.convert("L")
        arr = np.array(img).astype(np.uint8)

        # light spectrogram-domain aug
        if random.random() < 0.7:
            shift = random.randint(-3, 3)
            arr = np.roll(arr, shift=shift, axis=1)  # time shift

        if random.random() < 0.25:
            shift = random.randint(-2, 2)
            arr = np.roll(arr, shift=shift, axis=0)  # freq shift

        # time mask
        if random.random() < 0.5:
            w = random.randint(2, max(2, arr.shape[1] // 8))
            x0 = random.randint(0, max(0, arr.shape[1] - w))
            arr[:, x0:x0 + w] = 0

        # freq mask
        if random.random() < 0.5:
            h = random.randint(2, max(2, arr.shape[0] // 6))
            y0 = random.randint(0, max(0, arr.shape[0] - h))
            arr[y0:y0 + h, :] = 0

        img = Image.fromarray(arr, mode="L")
        img = img.resize((self.img_size, self.img_size), resample=Image.BICUBIC)

        # mild brightness/contrast
        if random.random() < 0.4:
            x = np.array(img).astype(np.float32) / 255.0
            scale = random.uniform(0.9, 1.15)
            bias = random.uniform(-0.04, 0.04)
            x = np.clip(x * scale + bias, 0.0, 1.0)
            img = Image.fromarray((x * 255).astype(np.uint8), mode="L")

        img = Image.merge("RGB", (img, img, img))
        x = self.to_tensor(img)
        x = self.normalize(x)
        return x


class SpectrogramEvalTransform:
    def __init__(self, img_size=224):
        self.img_size = img_size
        self.to_tensor = transforms.ToTensor()
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        )

    def __call__(self, img: Image.Image):
        img = img.convert("L")
        img = img.resize((self.img_size, self.img_size), resample=Image.BICUBIC)
        img = Image.merge("RGB", (img, img, img))
        x = self.to_tensor(img)
        x = self.normalize(x)
        return x


train_tf = SpectrogramTrainTransform(IMG_SIZE)
eval_tf = SpectrogramEvalTransform(IMG_SIZE)


# =========================================================
# 4. Dataset
# =========================================================
class FullTrainDataset:
    def __init__(self, root_dir: Path):
        self.root_dir = root_dir
        self.classes = sorted([p.name for p in root_dir.iterdir() if p.is_dir()])
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.idx_to_class = {i: c for c, i in self.class_to_idx.items()}

        self.samples = []
        for cls_name in self.classes:
            for img_path in sorted((root_dir / cls_name).rglob("*.png")):
                self.samples.append((img_path, self.class_to_idx[cls_name]))

        if not self.samples:
            raise RuntimeError(f"No PNG files found under {root_dir}")

        self.labels = [y for _, y in self.samples]


class TrainValDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        with Image.open(img_path) as img:
            x = self.transform(img)
        return x, label


class TestDataset(Dataset):
    def __init__(self, test_dir: Path, transform):
        self.image_paths = sorted(test_dir.rglob("*.png"))
        self.transform = transform
        if not self.image_paths:
            raise RuntimeError(f"No PNG files found under {test_dir}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        with Image.open(img_path) as img:
            x = self.transform(img)
        return x, img_path.name


full_train = FullTrainDataset(TRAIN_DIR)
num_classes = len(full_train.classes)

solution_df = pd.read_csv(SOLUTION_CSV)
solution_map = dict(zip(solution_df["file_name"], solution_df["label"]))

print("num_classes =", num_classes)
print("num_train =", len(full_train.samples))


# =========================================================
# 5. Model builders
# =========================================================
def build_model(model_name: str, num_classes: int):
    if model_name == "convnext_small":
        weights = models.ConvNeXt_Small_Weights.DEFAULT
        model = models.convnext_small(weights=weights)
        in_features = model.classifier[2].in_features
        model.classifier[2] = nn.Linear(in_features, num_classes)
        head_modules = [model.classifier]
    elif model_name == "efficientnet_v2_s":
        weights = models.EfficientNet_V2_S_Weights.DEFAULT
        model = models.efficientnet_v2_s(weights=weights)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)
        head_modules = [model.classifier]
    else:
        raise ValueError(model_name)

    return model, head_modules


def freeze_backbone_except_head(model, head_modules):
    for p in model.parameters():
        p.requires_grad = False
    for m in head_modules:
        for p in m.parameters():
            p.requires_grad = True


def unfreeze_all(model):
    for p in model.parameters():
        p.requires_grad = True


def make_optimizer(model, lr):
    return torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr,
        weight_decay=WEIGHT_DECAY,
    )


# =========================================================
# 6. Train / eval / predict
# =========================================================
@torch.no_grad()
def eval_epoch(model, loader, criterion, use_tta=False):
    model.eval()
    total_loss = 0.0
    all_logits = []
    all_labels = []
    use_amp = USE_AMP and DEVICE.type == "cuda"

    for x, y in tqdm(loader, leave=False):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        if use_tta:
            variants = [
                x,
                torch.roll(x, shifts=4, dims=3),
                torch.roll(x, shifts=-4, dims=3),
                torch.roll(x, shifts=2, dims=2),
                torch.roll(x, shifts=-2, dims=2),
            ]
            logits_sum = 0.0
            for v in variants:
                with torch.autocast(device_type="cuda", enabled=use_amp):
                    logits_sum = logits_sum + model(v)
            logits = logits_sum / len(variants)
        else:
            with torch.autocast(device_type="cuda", enabled=use_amp):
                logits = model(x)

        loss = criterion(logits, y)
        total_loss += loss.item() * x.size(0)
        all_logits.append(logits.float().cpu())
        all_labels.append(y.cpu())

    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()
    preds = logits.argmax(axis=1)
    acc = accuracy_score(labels, preds)
    return total_loss / len(loader.dataset), acc, logits, labels


def train_epoch(model, ema, loader, optimizer, scaler, criterion):
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    use_amp = USE_AMP and DEVICE.type == "cuda"

    for x, y in tqdm(loader, leave=False):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        if USE_MIXUP:
            x, y_a, y_b, lam = mixup_batch(x, y, MIXUP_ALPHA)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", enabled=use_amp):
            logits = model(x)
            if USE_MIXUP:
                loss = mixup_loss(criterion, logits, y_a, y_b, lam)
            else:
                loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()
        ema.update(model)

        total_loss += loss.item() * x.size(0)
        all_preds.extend(logits.argmax(dim=1).detach().cpu().numpy())
        all_labels.extend(y.detach().cpu().numpy())

    return total_loss / len(loader.dataset), accuracy_score(all_labels, all_preds)


@torch.no_grad()
def predict_logits(model, loader, use_tta=False):
    model.eval()
    all_logits = []
    all_names = []
    use_amp = USE_AMP and DEVICE.type == "cuda"

    for x, names in tqdm(loader, leave=False):
        x = x.to(DEVICE, non_blocking=True)

        if use_tta:
            variants = [
                x,
                torch.roll(x, shifts=4, dims=3),
                torch.roll(x, shifts=-4, dims=3),
                torch.roll(x, shifts=2, dims=2),
                torch.roll(x, shifts=-2, dims=2),
            ]
            logits_sum = 0.0
            for v in variants:
                with torch.autocast(device_type="cuda", enabled=use_amp):
                    logits_sum = logits_sum + model(v)
            logits = logits_sum / len(variants)
        else:
            with torch.autocast(device_type="cuda", enabled=use_amp):
                logits = model(x)

        all_logits.append(logits.float().cpu())
        all_names.extend(list(names))

    return torch.cat(all_logits, dim=0).numpy(), all_names


# =========================================================
# 7. CV training for one model
# =========================================================
def run_model_cv(model_name: str, lr_head: float, lr_backbone: float):
    print("\n" + "=" * 120)
    print(f"Running model: {model_name}")

    model_dir = OUTPUT_DIR / model_name
    model_dir.mkdir(parents=True, exist_ok=True)

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    oof_logits = np.zeros((len(full_train.samples), num_classes), dtype=np.float32)

    test_dataset = TestDataset(TEST_DIR, eval_tf)
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=(NUM_WORKERS > 0),
    )

    test_logits_accum = None
    test_file_order = None
    fold_summaries = []

    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(num_classes),
        y=np.array(full_train.labels),
    )
    class_weights = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

    for fold, (tr_idx, va_idx) in enumerate(
        skf.split(np.arange(len(full_train.samples)), full_train.labels), start=1
    ):
        print("\n" + "-" * 100)
        print(f"{model_name} | Fold {fold}/{N_SPLITS}")

        fold_dir = model_dir / f"fold_{fold}"
        fold_dir.mkdir(parents=True, exist_ok=True)

        train_samples = [full_train.samples[i] for i in tr_idx]
        val_samples = [full_train.samples[i] for i in va_idx]

        train_ds = TrainValDataset(train_samples, train_tf)
        val_ds = TrainValDataset(val_samples, eval_tf)

        train_loader = DataLoader(
            train_ds,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY,
            persistent_workers=(NUM_WORKERS > 0),
        )
        val_loader = DataLoader(
            val_ds,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY,
            persistent_workers=(NUM_WORKERS > 0),
        )

        model, head_modules = build_model(model_name, num_classes)
        model = model.to(DEVICE)
        ema = ModelEMA(model, decay=EMA_DECAY)

        criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

        freeze_backbone_except_head(model, head_modules)
        optimizer = make_optimizer(model, lr_head)
        scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and DEVICE.type == "cuda"))
        scheduler = torch.optim.lr_scheduler.LambdaLR(
            optimizer, lr_lambda=lambda ep: cosine_lr_lambda(ep, EPOCHS, warmup_epochs=1)
        )

        best_val_acc = -1.0
        best_epoch = -1
        best_state = None
        best_val_logits = None
        bad_epochs = 0
        history = []

        for epoch in range(1, EPOCHS + 1):
            if epoch == FREEZE_BACKBONE_EPOCHS + 1:
                print(f"[{model_name}][fold {fold}] unfreezing backbone")
                unfreeze_all(model)
                optimizer = make_optimizer(model, lr_backbone)
                scheduler = torch.optim.lr_scheduler.LambdaLR(
                    optimizer,
                    lr_lambda=lambda ep: cosine_lr_lambda(ep, max(EPOCHS - epoch + 1, 1), warmup_epochs=1)
                )

            print(f"\n[{model_name}][fold {fold}] epoch {epoch}/{EPOCHS}")

            train_loss, train_acc = train_epoch(model, ema, train_loader, optimizer, scaler, criterion)
            val_loss, val_acc, val_logits, val_labels = eval_epoch(
                ema.ema, val_loader, criterion, use_tta=USE_TTA
            )

            history.append({
                "epoch": epoch,
                "train_loss": float(train_loss),
                "train_acc": float(train_acc),
                "val_loss": float(val_loss),
                "val_acc": float(val_acc),
                "lr": float(optimizer.param_groups[0]["lr"]),
            })

            print(
                f"[{model_name}][fold {fold}] "
                f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
                f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} "
                f"lr={optimizer.param_groups[0]['lr']:.2e}"
            )

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_epoch = epoch
                best_state = copy.deepcopy(ema.ema.state_dict())
                best_val_logits = val_logits.copy()
                bad_epochs = 0
            else:
                bad_epochs += 1

            scheduler.step()

            if bad_epochs >= PATIENCE:
                print(f"[{model_name}][fold {fold}] early stopping")
                break

        ema.ema.load_state_dict(best_state)
        oof_logits[va_idx] = best_val_logits

        fold_test_logits, fold_names = predict_logits(ema.ema, test_loader, use_tta=USE_TTA)
        np.save(fold_dir / "test_logits.npy", fold_test_logits)

        if test_logits_accum is None:
            test_logits_accum = fold_test_logits.astype(np.float64)
            test_file_order = fold_names
        else:
            if test_file_order != fold_names:
                raise RuntimeError("Test file order mismatch across folds.")
            test_logits_accum += fold_test_logits.astype(np.float64)

        fold_summary = {
            "fold": fold,
            "best_val_acc": float(best_val_acc),
            "best_epoch": int(best_epoch),
            "history": history,
        }
        fold_summaries.append(fold_summary)
        save_json(fold_summary, fold_dir / "metrics.json")

    oof_preds = oof_logits.argmax(axis=1)
    oof_acc = accuracy_score(full_train.labels, oof_preds)

    test_logits = test_logits_accum / N_SPLITS
    pred_idx = test_logits.argmax(axis=1)
    pred_labels = [full_train.idx_to_class[int(i)] for i in pred_idx]

    y_true_test = [solution_map[f] for f in test_file_order]
    local_test_acc = float(np.mean(np.array(y_true_test) == np.array(pred_labels)))

    summary = {
        "model_name": model_name,
        "oof_accuracy": float(oof_acc),
        "local_test_accuracy": float(local_test_acc),
        "fold_summaries": fold_summaries,
    }
    save_json(summary, model_dir / "summary.json")

    print(f"\n{model_name} OOF accuracy: {oof_acc:.4f}")
    print(f"{model_name} local test accuracy: {local_test_acc:.4f}")

    return {
        "model_name": model_name,
        "oof_accuracy": float(oof_acc),
        "local_test_accuracy": float(local_test_acc),
        "test_logits": test_logits,
        "test_file_order": test_file_order,
    }


# =========================================================
# 8. Train both models and ensemble
# =========================================================
results = []
for spec in MODEL_SPECS:
    result = run_model_cv(
        model_name=spec["name"],
        lr_head=spec["lr_head"],
        lr_backbone=spec["lr_backbone"],
    )
    results.append(result)

# combine model logits
reference_order = results[0]["test_file_order"]
for r in results[1:]:
    if r["test_file_order"] != reference_order:
        raise RuntimeError("Model-level test file order mismatch.")

ensemble_logits = np.mean(np.stack([r["test_logits"] for r in results], axis=0), axis=0)
ensemble_preds = ensemble_logits.argmax(axis=1)
ensemble_labels = [full_train.idx_to_class[int(i)] for i in ensemble_preds]

submission_df = pd.DataFrame({
    "file_name": reference_order,
    "label": ensemble_labels,
})
submission_path = OUTPUT_DIR / "ensemble_submission.csv"
submission_df.to_csv(submission_path, index=False)

y_true_test = [solution_map[f] for f in reference_order]
ensemble_test_acc = float(np.mean(np.array(y_true_test) == np.array(ensemble_labels)))

summary = {
    "models": [
        {
            "model_name": r["model_name"],
            "oof_accuracy": r["oof_accuracy"],
            "local_test_accuracy": r["local_test_accuracy"],
        }
        for r in results
    ],
    "ensemble_local_test_accuracy": float(ensemble_test_acc),
    "submission_csv": str(submission_path),
}
save_json(summary, OUTPUT_DIR / "ensemble_summary.json")

print("\n" + "=" * 120)
for r in results:
    print(
        f"{r['model_name']}: "
        f"oof={r['oof_accuracy']:.4f} "
        f"local_test={r['local_test_accuracy']:.4f}"
    )
print(f"ensemble_local_test_accuracy={ensemble_test_acc:.4f}")
print(f"Saved submission to: {submission_path}")

submission_df.head()

Using device: cuda
GPU: NVIDIA A100-SXM4-40GB
num_classes = 9
num_train = 21898

Running model: convnext_small

----------------------------------------------------------------------------------------------------
convnext_small | Fold 1/5

[convnext_small][fold 1] epoch 1/16


/tmp/ipykernel_2308/1362803534.py:172: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(arr, mode="L")
/tmp/ipykernel_2308/1362803534.py:172: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(arr, mode="L")
/tmp/ipykernel_2308/1362803534.py:172: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(arr, mode="L")


  0%|          | 0/274 [00:00<?, ?it/s]

/tmp/ipykernel_2308/1362803534.py:181: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray((x * 255).astype(np.uint8), mode="L")
/tmp/ipykernel_2308/1362803534.py:181: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray((x * 255).astype(np.uint8), mode="L")
/tmp/ipykernel_2308/1362803534.py:181: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray((x * 255).astype(np.uint8), mode="L")
/tmp/ipykernel_2308/1362803534.py:172: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(arr, mode="L")
/tmp/ipykernel_2308/1362803534.py:181: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray((x * 255).astype(np.uint8), mode="L")


  0%|          | 0/69 [00:00<?, ?it/s]

...

  0%|          | 0/69 [00:00<?, ?it/s]

[efficientnet_v2_s][fold 5] train_loss=1.5785 train_acc=0.3401 val_loss=1.5847 val_acc=0.4492 lr=0.00e+00


  0%|          | 0/86 [00:00<?, ?it/s]


efficientnet_v2_s OOF accuracy: 0.4423
efficientnet_v2_s local test accuracy: 0.4523

convnext_small: oof=0.5144 local_test=0.5281
efficientnet_v2_s: oof=0.4423 local_test=0.4523
ensemble_local_test_accuracy=0.5128
Saved submission to: /content/drive/MyDrive/kaggle_cs3780_sp26/cnn_spectrogram_ensemble/ensemble_submission.csv


,file_name,label
0,1.png,Nocturnal bird
1,10.png,Water-associated bird
2,100.png,Flycatcher
3,1000.png,Nocturnal bird
4,1001.png,Flycatcher
